In [ ]:
import os

# used for configuring biogeme use of GPU, unused
# os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.95"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["TF_GPU_ALLOCATOR"] = "cuda_malloc_async"
# os.environ["CUDA_VISIBLE_DEVICES"] = ""

# Destination-choice + stay/move model via Larch

Larch port of `us/modeling_mnl.ipynb` (the Biogeme model), alongside `modeling_torch_choice.ipynb`
(the torch-choice port). Same choice structure: `alt=0` is staying, `alt=1..50` are the move
alternatives, and the data-prep cells below (reading, dtype cleanup, `ALT_CHOICE`, the melt to long
format, and the same-named-shared-column trick for tying coefficients across stay/move contexts) are
identical to the torch-choice port -- only the final "build the model and estimate it" section differs,
since that's the part where the two packages' APIs diverge. See `modeling_torch_choice.ipynb`'s intro
cell for the full list of fixes applied relative to `modeling_mnl.ipynb` as currently written on disk
(stale `OWN_RACE_PCT`/`OWN_GROUP_PCT` column names, a mangled NAICS variable name, the omitted
white/other-race terms, `ALT{i}_STATE` dtype).

**Larch's formula system (`P("name") * X("expression")`) makes the two awkward parts of the
torch-choice port unnecessary:**
- **Shared coefficients across stay/move contexts** are just `P("name")` referenced in the same
  `utility_ca` formula that gets evaluated once per `(person, alt)` row -- no need for the "same
  column name" trick to fake it (though the data is still built that way below, since the stay-context
  and move-context expressions are still different data, just tied to the same coefficient either way).
- **The un-parameterized `log(population)` offset** (coefficient pinned at 1, not estimated) is
  `m.lock_value("log_pop_offset", 1)` -- a native "holdfast" mechanism, not a `forward()` subclass hack.

**Package status, from the "experimental, not feature-complete" warning larch prints on import:** this
is `larch` v6 (the JAX/numba dual-backend rewrite), not the older, more battle-tested v5 line. Verified
empirically while porting -- **`m.utility_ca` accumulation via `+=` in a loop silently drops all but the
last term** (`m.utility_ca = PX(a); m.utility_ca += PX(b)` ends up as an *empty* `LinearFunction`, and
the model then estimates zero free parameters without erroring). Building the full sum on a plain local
variable first, then assigning it to `m.utility_ca` once, works correctly -- that's what's done below.
This isn't documented anywhere obvious, so worth flagging if you extend this notebook.

**Optimizer:** `maximize_loglike(method="BHHH")` -- Berndt-Hall-Hall-Hausman, a quasi-Newton method
built around the log-likelihood's own outer-product-of-gradients Hessian approximation. It's Larch's
default for unconstrained problems (SLSQP is the default only when finite parameter bounds/constraints
are set, which this model doesn't use) and is standard for MLE of logit-family models -- much closer in
spirit to Biogeme's BFGS than torch-choice's installed-version-limited Adam.


In [ ]:
import os
import sys

import larch as lx
import numpy as np
import pandas as pd
from larch import PX

sys.path.insert(0, os.path.abspath(".."))
from lib import io as lio
from lib import util as lut


In [ ]:
year = 2018
num_alternatives = 50


### Read data

In [ ]:
df_train = lio.read_estdata(
    year=year,
    num_alternatives=num_alternatives,
)
print(df_train.shape)

### Reshape to long format, build the Larch dataset

`lib.util.build_long_data` builds the long `(person_id, alt)` table shared by this notebook and
`modeling_torch_choice.ipynb`: `alt=0` is staying, `alt=1..num_alternatives` are the move alternatives, with the
stay/move-context values for shared coefficients (e.g. `proportion_same_age_18_34`) written under the
same column name so they tie to one coefficient downstream, and `log_pop_offset` (destination
population on move rows, origin population on the stay row) left as an un-parameterized term. See its
docstring for the full column-by-column breakdown, including the two `modeling_mnl.ipynb` race terms
omitted for the same reasons noted in this notebook's intro cell.

The returned `long_df` is already sorted by `(person_id, alt)`; `Dataset.construct.from_idca` takes it
directly (indexed by `(caseid, altid)`) -- no manual reshape into arrays needed, unlike the torch-choice
port.


In [ ]:
long_df, STAY_ONLY_TERMS, SHARED_TERMS, MOVE_ONLY_TERMS = lut.build_long_data(
    df_train, num_alternatives
)
varnames = STAY_ONLY_TERMS + SHARED_TERMS + MOVE_ONLY_TERMS

num_persons = df_train["person_id"].nunique()
num_alts = num_alternatives + 1  # 51: alt=0 (stay) + alt=1..50 (move)

idca = long_df.set_index(["person_id", "alt"])[["choice", "log_pop_offset"] + varnames]
ds = lx.Dataset.construct.from_idca(idca, crack=True)
ds


### Setting up the model

One `P(name) * X(name)` term per `varnames` entry via the `PX` shorthand, summed on a plain local
variable and assigned to `m.utility_ca` once at the end -- **not** built with `m.utility_ca += ...` in
a loop, which silently discards everything but the last term (see intro cell). No separate ASC/intercept
term is added -- `stay` in `STAY_ONLY_TERMS` is already an explicit ASC for staying, matching
`fit_intercept=False` in the torch-choice port / Biogeme not adding an implicit ASC of its own.

`log_pop_offset` gets a coefficient too, then `m.lock_value("log_pop_offset", 1)` pins it at exactly 1
(`holdfast`), reproducing Biogeme's bare `log(Variable(...))` calls (and xlogit's `addit=`) without
needing a custom subclass the way the torch-choice port did.


In [ ]:
m = lx.Model(ds)
m.title = f"us_mnl_{year} (larch port of modeling_mnl.ipynb)"
m.compute_engine = "numba"

total_utility = PX(varnames[0])
for name in varnames[1:]:
    total_utility = total_utility + PX(name)
total_utility = total_utility + PX("log_pop_offset")
m.utility_ca = total_utility

m.choice_ca_var = "choice"
# all alternatives are available for everyone (matches av[i] = 1 for all i in modeling_mnl.ipynb);
# no availability_ca_var needed.

m.lock_value("log_pop_offset", 1)

m.ordering = [
    ("Stay", "stay.*"),
    ("Shared", "proportion.*|median_.*|unemp_rate|vacancy_rate"),
    ("Destination-only", "destchoice.*"),
    ("Offset", "log_pop_offset"),
]


### Fitting

In [ ]:
print("null log-likelihood:", m.loglike())


In [ ]:
result = m.maximize_loglike(method="BHHH")
result


In [ ]:
m.calculate_parameter_covariance()
m.parameter_summary()
